# Fire2Air Darwin — Assessment 3
## 04 Model 3 Count Prediction and Explainability
**Member:** Navodya Piumanthi

Corrected notebook generated from the new Assessment 3 Model 3 source code.
The cells are ordered so that `build_count_model`, `candidate_specs`,
`best_model_name`, and `selected_params` are defined before they are used.


In [2]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor, TweedieRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_poisson_deviance
)

# Optional XGBoost
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception as exc:
    HAS_XGBOOST = False
    print("XGBoost unavailable:", exc)

# Optional SHAP
try:
    import shap
    HAS_SHAP = True
except Exception as exc:
    HAS_SHAP = False
    print("SHAP unavailable:", exc)

RANDOM_STATE = 42
BOOTSTRAP_REPEATS = 300

possible_source_folders = [
    Path.cwd(),
    Path.cwd() / "Source code",
    Path.cwd().parent / "Source code",
]

source_folder = next(
    (
        folder.resolve()
        for folder in possible_source_folders
        if (
            folder
            / "outputs_prt661"
            / "processed"
            / "Fire2Air_model_ready_checkpoint_A3.csv"
        ).exists()
    ),
    None,
)

if source_folder is None:
    raise FileNotFoundError(
        "Fire2Air_model_ready_checkpoint_A3.csv not found. "
        "Expected location: "
        "Source code/outputs_prt661/processed/"
        "Fire2Air_model_ready_checkpoint_A3.csv"
    )

output_root = source_folder / "outputs_prt661"

processed_dir = output_root / "processed"
metadata_dir = output_root / "metadata"
table_dir = output_root / "tables"
figure_dir = output_root / "figures"
model_dir = output_root / "models"

# Create output folders if they do not already exist
for folder in [table_dir, figure_dir, model_dir]:
    folder.mkdir(parents=True, exist_ok=True)

data_path = (
    processed_dir
    / "Fire2Air_model_ready_checkpoint_A3.csv"
)

integrated_daily = pd.read_csv(
    data_path,
    parse_dates=["date", "target_date"],
)

manifest_path = (
    metadata_dir
    / "Fire2Air_feature_manifest_A3.json"
)

with open(
    manifest_path,
    "r",
    encoding="utf-8"
) as f:
    manifest = json.load(f)

model_features = manifest["features"]
feature_groups = manifest["feature_groups"]
duration_target = manifest["targets"]["count"]

print("Source folder:", source_folder)
print("Dataset:", data_path)
print("Manifest:", manifest_path)
print("Dataset shape:", integrated_daily.shape)
print("Target:", duration_target)


Source folder: C:\Users\navod\Desktop\PRT661---DATA-SCIENCE-PRACTICE---Dan5---Theme2\Source code
Dataset: C:\Users\navod\Desktop\PRT661---DATA-SCIENCE-PRACTICE---Dan5---Theme2\Source code\outputs_prt661\processed\Fire2Air_model_ready_checkpoint_A3.csv
Manifest: C:\Users\navod\Desktop\PRT661---DATA-SCIENCE-PRACTICE---Dan5---Theme2\Source code\outputs_prt661\metadata\Fire2Air_feature_manifest_A3.json
Dataset shape: (7530, 120)
Target: target_hours_ge_25_next_day


## A. Chronological split and target distribution


In [3]:
duration_data = integrated_daily.dropna(subset=[duration_target, "target_date"]).copy()
duration_data["target_year"] = duration_data["target_date"].dt.year

train_duration = duration_data[duration_data["target_year"] <= 2022].copy()
validation_duration = duration_data[duration_data["target_year"] == 2023].copy()
test_duration = duration_data[duration_data["target_year"] == 2024].copy()

if min(len(train_duration), len(validation_duration), len(test_duration)) == 0:
    raise ValueError(
        f"Insufficient chronological data: train={len(train_duration)}, "
        f"validation={len(validation_duration)}, test={len(test_duration)}"
    )
assert train_duration["target_date"].max() < validation_duration["target_date"].min()
assert validation_duration["target_date"].max() < test_duration["target_date"].min()
assert duration_data[duration_target].between(0, 24).all()

X_train, y_train = train_duration[model_features], train_duration[duration_target]
X_validation, y_validation = validation_duration[model_features], validation_duration[duration_target]
X_test, y_test = test_duration[model_features], test_duration[duration_target]


## B. Preprocessing, model builders and metrics


In [4]:
def onehot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(features, scale_numeric=False):
    categorical = [f for f in features if f == "station"]
    numeric = [f for f in features if f not in categorical]
    transformers = []
    if categorical:
        transformers.append((
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", onehot()),
            ]),
            categorical,
        ))
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        steps.append(("scaler", StandardScaler()))
    if numeric:
        transformers.append(("numeric", Pipeline(steps), numeric))
    return ColumnTransformer(transformers)


def bounded(predictions):
    return np.clip(np.asarray(predictions, dtype=float), 0, 24)


def count_metrics(y_true, predictions):
    y = np.asarray(y_true, dtype=float)
    pred = bounded(predictions)
    return {
        "MAE": mean_absolute_error(y, pred),
        "RMSE": float(np.sqrt(mean_squared_error(y, pred))),
        "R2": r2_score(y, pred) if len(y) >= 2 else np.nan,
        "Mean_Poisson_Deviance": mean_poisson_deviance(y, np.clip(pred, 1e-6, None)),
        "Prediction_Min": float(pred.min()),
        "Prediction_Max": float(pred.max()),
    }


def duration_strata_metrics(y_true, predictions):
    y = np.asarray(y_true, dtype=float)
    pred = bounded(predictions)
    strata = [
        ("Zero hours", y == 0),
        ("Low duration 1-5 h", (y >= 1) & (y <= 5)),
        ("Medium duration 6-12 h", (y >= 6) & (y <= 12)),
        ("Long duration 13-24 h", y >= 13),
    ]
    rows = []
    for label, mask in strata:
        if mask.sum() > 0:
            rows.append({
                "Stratum": label,
                "Rows": int(mask.sum()),
                "MAE": mean_absolute_error(y[mask], pred[mask]),
                "RMSE": float(np.sqrt(mean_squared_error(y[mask], pred[mask]))),
                "Bias_PredMinusObs": float(np.mean(pred[mask] - y[mask])),
            })
    return rows


class HurdleRandomForest(BaseEstimator, RegressorMixin):
    """Exploratory two-stage count model.

    Stage 1 predicts whether any elevated hour occurs. Stage 2 predicts duration on
    positive-duration days. The expected duration is P(any) * predicted positive hours.
    """
    def __init__(self, features, n_estimators=300, max_depth=None, min_samples_leaf=2, random_state=42):
        self.features = features
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state

    def fit(self, X, y):
        self.preprocessor_ = build_preprocessor(self.features, scale_numeric=False)
        Xt = self.preprocessor_.fit_transform(X[self.features])
        y_arr = np.asarray(y, dtype=float)
        self.classifier_ = RandomForestClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_leaf=self.min_samples_leaf,
            class_weight="balanced",
            random_state=self.random_state,
            n_jobs=-1,
        )
        self.classifier_.fit(Xt, (y_arr > 0).astype(int))
        positive = y_arr > 0
        self.regressor_ = RandomForestRegressor(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_leaf=self.min_samples_leaf,
            random_state=self.random_state,
            n_jobs=-1,
        )
        if positive.sum() > 0:
            self.regressor_.fit(Xt[positive], y_arr[positive])
            self.positive_mean_ = float(np.mean(y_arr[positive]))
        else:
            self.positive_mean_ = 1.0
        return self

    def predict(self, X):
        Xt = self.preprocessor_.transform(X[self.features])
        p_any = self.classifier_.predict_proba(Xt)[:, 1]
        try:
            positive_hours = self.regressor_.predict(Xt)
        except Exception:
            positive_hours = np.repeat(self.positive_mean_, len(X))
        return bounded(p_any * np.clip(positive_hours, 0, 24))


def build_count_model(model_name, params, features):
    if model_name == "Poisson Regression":
        return Pipeline([
            ("preprocessor", build_preprocessor(features, scale_numeric=True)),
            ("model", PoissonRegressor(alpha=params["alpha"], max_iter=1000)),
        ])
    if model_name == "Tweedie Regression":
        return Pipeline([
            ("preprocessor", build_preprocessor(features, scale_numeric=True)),
            ("model", TweedieRegressor(power=1.5, alpha=params["alpha"], link="log", max_iter=1000)),
        ])
    if model_name == "Random Forest":
        return Pipeline([
            ("preprocessor", build_preprocessor(features, scale_numeric=False)),
            ("model", RandomForestRegressor(
                n_estimators=300,
                max_depth=params["max_depth"],
                min_samples_leaf=params["min_samples_leaf"],
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])
    if model_name == "XGBoost Poisson":
        if not HAS_XGBOOST:
            raise RuntimeError("XGBoost is unavailable.")
        return Pipeline([
            ("preprocessor", build_preprocessor(features, scale_numeric=False)),
            ("model", XGBRegressor(
                n_estimators=params["n_estimators"],
                learning_rate=params["learning_rate"],
                max_depth=params["max_depth"],
                objective="count:poisson",
                eval_metric="poisson-nloglik",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])
    if model_name == "Hurdle Random Forest":
        return HurdleRandomForest(
            features=features,
            n_estimators=300,
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=RANDOM_STATE,
        )
    raise ValueError(f"Unknown count model: {model_name}")


## C. Candidate models and validation tuning


In [5]:
candidate_specs = [
    ("Poisson Regression", {"alpha": 0.1}, "alpha=0.1"),
    ("Poisson Regression", {"alpha": 1.0}, "alpha=1.0"),
    ("Tweedie Regression", {"alpha": 0.1}, "power=1.5;alpha=0.1"),
    ("Tweedie Regression", {"alpha": 1.0}, "power=1.5;alpha=1.0"),
]
for max_depth, min_samples_leaf in [(None, 1), (None, 4), (10, 4), (15, 4)]:
    candidate_specs.append((
        "Random Forest",
        {"max_depth": max_depth, "min_samples_leaf": min_samples_leaf},
        f"max_depth={max_depth};min_samples_leaf={min_samples_leaf}",
    ))
# One transparent two-stage benchmark to investigate the Assessment 3 recommendation.
candidate_specs.append((
    "Hurdle Random Forest",
    {"max_depth": 10, "min_samples_leaf": 4},
    "two-stage:any-event+positive-duration",
))
if HAS_XGBOOST:
    for n_estimators, learning_rate, max_depth in [
        (100, 0.05, 2), (200, 0.05, 2), (300, 0.05, 2), (200, 0.05, 3)
    ]:
        candidate_specs.append((
            "XGBoost Poisson",
            {"n_estimators": n_estimators, "learning_rate": learning_rate, "max_depth": max_depth},
            f"n_estimators={n_estimators};learning_rate={learning_rate};max_depth={max_depth}",
        ))

validation_rows = []
model_templates = {}
params_lookup = {}
for model_name, params, config_id in candidate_specs:
    model = build_count_model(model_name, params, model_features)
    model.fit(X_train, y_train)
    raw_pred = model.predict(X_validation)
    validation_rows.append({
        "Model": model_name,
        "Config": config_id,
        "Raw_Negative_Predictions": int((np.asarray(raw_pred) < 0).sum()),
        **count_metrics(y_validation, raw_pred),
    })
    model_templates[(model_name, config_id)] = model
    params_lookup[(model_name, config_id)] = params

validation_results = pd.DataFrame(validation_rows).sort_values(
    ["MAE", "RMSE", "Mean_Poisson_Deviance"]
).reset_index(drop=True)
validation_results.to_csv(table_dir / "a3_model3_count_validation_results.csv", index=False)
print("\n2023 count validation results:")
print(validation_results.round(4).to_string(index=False))

selected = validation_results.iloc[0]
best_model_name = selected["Model"]
best_config_id = selected["Config"]
selected_params = params_lookup[(best_model_name, best_config_id)]
selected_template = model_templates[(best_model_name, best_config_id)]
print("\nSelected count model by validation MAE:", best_model_name)
print("Configuration:", best_config_id)



2023 count validation results:
               Model                                          Config  Raw_Negative_Predictions    MAE   RMSE     R2  Mean_Poisson_Deviance  Prediction_Min  Prediction_Max
  Poisson Regression                                       alpha=0.1                         0 2.2377 3.6821 0.5821                 3.2100          0.0125         24.0000
  Tweedie Regression                             power=1.5;alpha=0.1                         0 2.2523 3.8112 0.5523                 3.2461          0.0078         24.0000
  Poisson Regression                                       alpha=1.0                         0 2.3820 3.8604 0.5406                 3.2758          0.0728         24.0000
       Random Forest                 max_depth=10;min_samples_leaf=4                         0 2.4298 4.0159 0.5029                 3.3071          0.0272         12.4798
       Random Forest                 max_depth=15;min_samples_leaf=4                         0 2.4367 4.0216 0.50

## D. A–F ablation study


In [6]:
ablation_rows = []
for stage, stage_features in feature_groups.items():
    stage_features = [f for f in stage_features if f in duration_data.columns]
    model = build_count_model(best_model_name, selected_params, stage_features)
    model.fit(train_duration[stage_features], y_train)
    pred = model.predict(validation_duration[stage_features])
    ablation_rows.append({
        "Stage": stage,
        "Feature_Count": len(stage_features),
        **count_metrics(y_validation, pred),
    })
ablation_results = pd.DataFrame(ablation_rows)
ablation_results.to_csv(table_dir / "a3_model3_ablation_results.csv", index=False)


## E. Final 2024 evaluation


In [7]:
development_duration = pd.concat([train_duration, validation_duration], ignore_index=True)
X_development = development_duration[model_features]
y_development = development_duration[duration_target]

# Rebuild rather than clone custom hurdle models to avoid estimator-state ambiguity.
final_count_model = build_count_model(best_model_name, selected_params, model_features)
final_count_model.fit(X_development, y_development)
raw_test_predictions = final_count_model.predict(X_test)
test_predictions = bounded(raw_test_predictions)

final_metrics = {
    "Model": best_model_name,
    "Config": best_config_id,
    "Raw_Negative_Predictions": int((np.asarray(raw_test_predictions) < 0).sum()),
    **count_metrics(y_test, raw_test_predictions),
}
final_metrics_df = pd.DataFrame([final_metrics])
final_metrics_df.to_csv(table_dir / "a3_model3_final_2024_metrics.csv", index=False)
print("\nFinal 2024 count performance:")
print(final_metrics_df.round(4).to_string(index=False))

strata_results = pd.DataFrame(duration_strata_metrics(y_test, test_predictions))
strata_results.to_csv(table_dir / "a3_model3_2024_duration_strata.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, test_predictions, alpha=0.35, s=15)
ax.plot([0, 24], [0, 24], linestyle="--")
ax.set_xlim(0, 24)
ax.set_ylim(0, 24)
ax.set_xlabel("Observed elevated hours next day")
ax.set_ylabel("Predicted elevated hours next day")
ax.set_title(f"2024 Elevated-Hours Count — {best_model_name}")
fig.tight_layout()
fig.savefig(figure_dir / "a3_model3_2024_observed_vs_predicted.png", dpi=180, bbox_inches="tight")
plt.close(fig)



Final 2024 count performance:
             Model    Config  Raw_Negative_Predictions    MAE   RMSE     R2  Mean_Poisson_Deviance  Prediction_Min  Prediction_Max
Poisson Regression alpha=0.1                         0 1.3058 2.1642 0.4575                 1.8845          0.0153         13.5244


## F. Station-level and leave-one-station-out evaluation


In [8]:
station_rows = []
test_reset = test_duration.reset_index(drop=True)
y_reset = y_test.reset_index(drop=True)
for station, g in test_reset.groupby("station"):
    idx = g.index.to_numpy()
    station_rows.append({
        "Station": station,
        "Rows": len(idx),
        **count_metrics(y_reset.iloc[idx], test_predictions[idx]),
    })
station_metrics = pd.DataFrame(station_rows)
station_metrics.to_csv(table_dir / "a3_model3_2024_metrics_by_station.csv", index=False)

loso_rows = []
for held_out in sorted(duration_data["station"].dropna().unique()):
    train_loso = development_duration[development_duration["station"] != held_out].copy()
    test_loso = test_duration[test_duration["station"] == held_out].copy()
    if train_loso.empty or test_loso.empty:
        continue
    model = build_count_model(best_model_name, selected_params, model_features)
    model.fit(train_loso[model_features], train_loso[duration_target])
    pred = model.predict(test_loso[model_features])
    loso_rows.append({
        "Held_Out_Station": held_out,
        "Train_Stations": ", ".join(sorted(train_loso["station"].unique())),
        "Test_Rows_2024": len(test_loso),
        **count_metrics(test_loso[duration_target], pred),
    })
pd.DataFrame(loso_rows).to_csv(table_dir / "a3_model3_leave_one_station_out.csv", index=False)


## G. Explainability: SHAP where supported + permutation importance always


In [9]:
def dense_array(x):
    return x.toarray() if hasattr(x, "toarray") else np.asarray(x)


if HAS_SHAP and isinstance(final_count_model, Pipeline):
    try:
        prep = final_count_model.named_steps["preprocessor"]
        estimator = final_count_model.named_steps["model"]
        background_raw = X_development.sample(n=min(120, len(X_development)), random_state=RANDOM_STATE)
        explain_raw = X_test.sample(n=min(160, len(X_test)), random_state=RANDOM_STATE)
        background_t = dense_array(prep.transform(background_raw))
        explain_t = dense_array(prep.transform(explain_raw))
        feature_names = prep.get_feature_names_out()

        explainer = shap.Explainer(estimator, background_t, feature_names=feature_names)
        shap_values = explainer(explain_t)

        shap.plots.bar(shap_values, max_display=20, show=False)
        plt.title(f"SHAP Global Importance — Count Model ({best_model_name})")
        plt.tight_layout()
        plt.savefig(figure_dir / "a3_model3_shap_global_bar.png", dpi=180, bbox_inches="tight")
        plt.close()

        shap.plots.beeswarm(shap_values, max_display=20, show=False)
        plt.title(f"SHAP Summary — Count Model ({best_model_name})")
        plt.tight_layout()
        plt.savefig(figure_dir / "a3_model3_shap_beeswarm.png", dpi=180, bbox_inches="tight")
        plt.close()
    except Exception as exc:
        print("SHAP could not complete for the selected model/package combination:", repr(exc))
else:
    print("SHAP skipped for this selected model; permutation importance remains available.")

perm = permutation_importance(
    final_count_model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
perm_df = (
    pd.DataFrame({
        "feature": model_features,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
perm_df.to_csv(table_dir / "a3_model3_count_permutation_importance.csv", index=False)

top_perm = perm_df.head(20).sort_values("importance_mean")
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top_perm["feature"], top_perm["importance_mean"], xerr=top_perm["importance_std"])
ax.set_xlabel("Increase in MAE when feature is permuted")
ax.set_title(f"Permutation Importance — Count Model ({best_model_name})")
fig.tight_layout()
fig.savefig(figure_dir / "a3_model3_count_permutation_importance.png", dpi=180, bbox_inches="tight")
plt.close(fig)


Background dataset has 120 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=120 when initializing the masker.


## H. Bootstrap uncertainty


In [10]:
rng = np.random.default_rng(RANDOM_STATE)
y_arr = np.asarray(y_test)
p_arr = np.asarray(test_predictions)
bootstrap_rows = []
for _ in range(BOOTSTRAP_REPEATS):
    idx = rng.integers(0, len(y_arr), len(y_arr))
    m = count_metrics(y_arr[idx], p_arr[idx])
    bootstrap_rows.append({"MAE": m["MAE"], "RMSE": m["RMSE"], "Mean_Poisson_Deviance": m["Mean_Poisson_Deviance"]})
bootstrap_df = pd.DataFrame(bootstrap_rows)
ci_rows = []
for metric in ["MAE", "RMSE", "Mean_Poisson_Deviance"]:
    vals = bootstrap_df[metric].dropna()
    ci_rows.append({
        "Metric": metric,
        "CI_2.5%": vals.quantile(0.025),
        "CI_97.5%": vals.quantile(0.975),
        "Bootstrap_Repeats_Used": len(vals),
    })
pd.DataFrame(ci_rows).to_csv(table_dir / "a3_model3_bootstrap_95ci.csv", index=False)


## I. Save predictions and model


In [11]:
count_predictions = test_duration[["date", "target_date", "station", duration_target]].copy()
count_predictions["raw_prediction"] = raw_test_predictions
count_predictions["bounded_prediction_0_24"] = test_predictions
count_predictions.to_csv(table_dir / "a3_model3_2024_predictions.csv", index=False)

joblib.dump(final_count_model, model_dir / "a3_model3_count.joblib")
with open(model_dir / "a3_model3_count_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "model": best_model_name,
        "config": best_config_id,
        "params": selected_params,
        "primary_selection_metric": "MAE",
        "count_specific_metric": "Mean_Poisson_Deviance",
        "prediction_bounds": [0, 24],
        "features": model_features,
        "assessment3_additions": [
            "FSI and wind-aligned features",
            "two-stage hurdle benchmark",
            "duration-stratified evaluation",
            "A-F ablation",
            "station-level metrics",
            "leave-one-station-out validation",
            "SHAP/permutation importance",
            "bootstrap uncertainty",
        ],
    }, f, indent=2)

print("\nSaved Assessment 3 Model 3 outputs.")



Saved Assessment 3 Model 3 outputs.


In [12]:
print("MODEL 3 — 2024 DURATION-STRATIFIED EVALUATION")
display(strata_results.round(4))

print("\nMODEL 3 — 2024 STATION-LEVEL EVALUATION")
display(station_metrics.round(4))

MODEL 3 — 2024 DURATION-STRATIFIED EVALUATION


,Stratum,Rows,MAE,RMSE,Bias_PredMinusObs
0,Zero hours,457,0.7396,1.2143,0.7396
1,Low duration 1-5 h,109,1.7473,2.3799,0.7371
2,Medium duration 6-12 h,74,3.6778,4.2540,-2.9312
3,Long duration 13-24 h,5,8.3226,8.8383,-8.3226



MODEL 3 — 2024 STATION-LEVEL EVALUATION


,Station,Rows,MAE,RMSE,R2,Mean_Poisson_Deviance,Prediction_Min,Prediction_Max
0,Palmerston,170,1.5555,2.4382,0.3670,2.1816,0.0416,12.9870
1,Stokes Hill,264,1.1399,1.9556,0.5403,1.5297,0.0185,13.5244
2,Winnellie,211,1.3121,2.1778,0.4322,2.0890,0.0153,11.4944
